|<img src="https://www.udp.cl/cms/wp-content/uploads/2021/06/UDP_LogoRGB_2lineas_Color_SinFondo.png" width="500" height="100">| <p> Ciencias de Datos para la Economía</p>  <p> Ingeniería Comercial </p><p> Profesor: Luis Cuevas Parra </p><p>Unidad 4: Pre-procesamiento y Análisis Exploratorio de Indicadores</p><p> Limpieza, deflactación, estacionalidad y visualización de coyuntura con datos financieros </p>|
|---|---|

# Pre-procesamiento y análisis exploratorio de indicadores

Antes de estimar cualquier modelo, un analista pasa la mayor parte del tiempo **preparando los datos**:
completar o descartar datos faltantes, poner las series en la misma escala, quitar el efecto de la inflación
y entender qué mueve a cada serie. Un error en esta etapa contamina todo lo que viene después.

En este cuaderno trabajaremos con **datos financieros** (dólar, cobre, acciones, oro, petróleo) junto con los índices
de precios de Chile y Estados Unidos.

### Lo que aprenderemos

| # | Tema | Pregunta que responde |
|:-:|:---|:---|
| 1 | Datos | ¿De dónde salen y cómo los guardo? |
| 2 | Valores faltantes | ¿Por qué faltan datos y qué hago con ellos? |
| 3 | Cambios de base y empalmes | ¿Cómo comparo series en distintas unidades o con distinta base? |
| 4 | Nominal vs. real | ¿Cuánto de un cambio de precio es solo inflación? |
| 5 | Componentes y estacionalidad | ¿Todas las series tienen estacionalidad? |
| 6 | Visualización de coyuntura | ¿Cómo comparo contra el año anterior, quién explica un resultado y qué se mueve junto? |

### Mini-glosario

| Término | Significado en una línea |
|:---|:---|
| **Valor faltante** (`NaN`) | Una fecha sin dato: feriado, dato aún no publicado o error de captura. |
| **Índice / año base** | Número que expresa un nivel respecto a un período que vale 100. |
| **Empalme** | Unir una serie antigua y una nueva que tienen distinta base. |
| **Nominal** | Medido en dinero de cada momento (pesos o dólares "de ese día"). |
| **Real** | Medido en dinero de **poder de compra constante**: se le quitó la inflación. |
| **Deflactar** | Pasar de nominal a real, dividiendo por un índice de precios. |
| **Retorno** | Cambio porcentual del precio de un activo en un período. |
| **Contribución** | Parte de un resultado total que explica cada componente (peso × variación). |

## 0. Preparar el entorno

In [1]:
import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10.5,
                     'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

# Paleta de colores del curso
COLOR_UDP   = '#800020'   # serie principal
COLOR_TREND = '#1f77b4'   # tendencia / series reales
COLOR_OK    = '#2ca02c'   # lectura favorable
COLOR_ALERT = '#d62728'   # alertas
COLOR_ALT   = '#ff7f0e'   # comparación
COLOR_GRAY  = '#7f7f7f'   # series de fondo

MESES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

## 1. Datos

Trabajaremos con dos tablas: una **diaria** (precios de mercado) y una **mensual** (índices de precios y actividad).

| Nombre corto | Qué es | Fuente | Frecuencia |
|:---|:---|:---|:---|
| `dolar` | Dólar observado, pesos por dólar | Banco Central (base SQL del kit) | Diaria |
| `cobre` | Cobre en Nueva York (`HG=F`), US\$ por libra | Yahoo Finance | Diaria |
| `acciones_cl` | ETF de acciones chilenas en Nueva York (`ECH`), US\$ | Yahoo Finance | Diaria |
| `sp500` | Índice bursátil de EE.UU. (`^GSPC`), puntos | Yahoo Finance | Diaria |
| `oro` | Oro (`GC=F`), US\$ por onza | Yahoo Finance | Diaria |
| `brent` | Petróleo Brent (`BZ=F`), US\$ por barril | Yahoo Finance | Diaria |
| `ipc_canasta_2018`, `ipc_canasta_2023` | IPC de Chile de cada canasta, sin empalmar | Banco Central (`data/raw/ipc_canastas.db`) | Mensual |
| `ipc_cl` | IPC de Chile, serie empalmada oficial (2023 = 100) | Banco Central (`data/raw/ipc_canastas.db`) | Mensual |
| `ipc_eeuu` | IPC de Estados Unidos (`CPIAUCSL`) | FRED (base SQL del kit) | Mensual |
| `imacec` | Actividad económica de Chile (2018 = 100) | Banco Central (base SQL del kit) | Mensual |

**Yahoo Finance** no tiene una API oficial; usamos la librería **`yfinance`**, que descarga los precios en una línea
(si no la tienes: `pip install yfinance`). Al final del cuaderno, un anexo muestra cómo pedir los mismos datos "a mano" con `requests`.

Usamos **precios ajustados** (`auto_adjust=True`): incluyen los dividendos, así que reflejan lo que realmente ganó quien invirtió.

In [ ]:
# Las dos bases SQL están en la carpeta data/raw de este módulo
BASE_KIT      = 'sqlite:///../data/raw/coyuntura_2026-07-31.db'   # base del kit (Unidad 1)
BASE_CANASTAS = 'sqlite:///../data/raw/ipc_canastas.db'           # IPC por canasta

### 1.1 Precios de mercado desde Yahoo Finance

In [3]:
tickers = {'HG=F': 'cobre', 'ECH': 'acciones_cl', '^GSPC': 'sp500', 'GC=F': 'oro', 'BZ=F': 'brent'}

yahoo = yf.download(list(tickers), start='2016-10-01', auto_adjust=True, progress=False)
yahoo = yahoo['Close']                      # nos quedamos con el precio de cierre
yahoo = yahoo.rename(columns=tickers)       # nombres cortos
yahoo.tail()

Ticker,brent,acciones_cl,oro,cobre,sp500
Date,,,,,
2026-10-02,102.250000,37.099998,4162.299805,6.4920,7722.720215
2026-10-05,100.320000,38.000000,4156.799805,6.5855,7773.950195
2026-10-06,100.580002,38.310001,4187.100098,6.5945,7818.930176
2026-10-07,100.199997,37.430000,4140.700195,6.5965,7801.770020
2026-10-08,104.070000,NaN,4150.100098,6.6090,7773.209961


### 1.2 Series desde las bases SQL

Esta función lee **una** serie de una base SQLite del curso (lo visto en la Unidad 1). La usaremos varias veces.

In [ ]:
def leer_serie(base, codigo):
    consulta = f"SELECT fecha, valor FROM observaciones WHERE id_serie = '{codigo}'"
    tabla = pd.read_sql(consulta, base, parse_dates=['fecha'], index_col='fecha')
    return tabla['valor']

dolar = leer_serie(BASE_KIT, 'F073.TCO.PRE.Z.D')
dolar.tail()

### 1.3 Tabla diaria: juntar el dólar con los precios de Yahoo

`join(..., how='outer')` conserva **todas** las fechas de ambas tablas. Si un día hay dato en una y no en la otra,
queda un `NaN`: eso lo trabajamos en la sección 2.

In [5]:
diarios = yahoo.join(dolar.rename('dolar'), how='outer')
diarios.tail()

,brent,acciones_cl,oro,cobre,sp500,dolar
2026-10-02,102.250000,37.099998,4162.299805,6.4920,7722.720215,NaN
2026-10-05,100.320000,38.000000,4156.799805,6.5855,7773.950195,NaN
2026-10-06,100.580002,38.310001,4187.100098,6.5945,7818.930176,NaN
2026-10-07,100.199997,37.430000,4140.700195,6.5965,7801.770020,NaN
2026-10-08,104.070000,NaN,4150.100098,6.6090,7773.209961,NaN


### 1.4 Tabla mensual

In [ ]:
mensuales = pd.DataFrame({
    'ipc_canasta_2018': leer_serie(BASE_CANASTAS, 'F074.IPC.IND.Z.2018.C.M'),
    'ipc_canasta_2023': leer_serie(BASE_CANASTAS, 'F074.IPC.IND.Z.2023.C.M'),
    'ipc_cl':           leer_serie(BASE_CANASTAS, 'F074.IPC.IND.Z.EP23.C.M'),
    'ipc_eeuu':         leer_serie(BASE_KIT, 'CPIAUCSL'),
    'imacec':           leer_serie(BASE_KIT, 'F032.IMC.IND.Z.Z.EP18.Z.Z.0.M'),
})
mensuales.tail()

### 1.5 Guardar en Parquet

Como en el cuaderno U2-01: guardamos las tablas para poder leerlas después sin volver a descargar.

In [7]:
diarios.to_parquet('../data/diarios.parquet')
mensuales.to_parquet('../data/mensuales.parquet')

## 2. Valores faltantes

Al juntar series de distintas fuentes en una sola tabla, aparecen celdas vacías (`NaN`). Las causas típicas son:

| Causa | Ejemplo en nuestros datos |
|:---|:---|
| **Calendarios distintos** | El 18 y 19 de septiembre (Fiestas Patrias) no hay dólar observado, pero Nueva York abre. El 4 de julio pasa lo contrario. |
| **Series que empiezan en fechas distintas** | El dólar observado parte en 2010; las series de Yahoo, en 2016. |
| **Datos aún no publicados** | La base del kit llega hasta el 31-jul-2026; Yahoo, hasta octubre. |
| **Errores de captura** | Un precio que no se registró ese día (o uno absurdo, que conviene convertir a `NaN`). |

**Primer paso siempre: diagnosticar** cuántos faltan, dónde y por qué. Recién después se decide qué hacer.

In [8]:
diagnostico = pd.DataFrame({
    'primer dato':  diarios.apply(lambda s: s.first_valid_index()),
    'último dato':  diarios.apply(lambda s: s.last_valid_index()),
    'faltantes':    diarios.isna().sum(),
    '% faltante':   (diarios.isna().mean() * 100).round(1),
})
diagnostico

,primer dato,último dato,faltantes,% faltante
brent,2016-10-03,2026-10-08,1755,41.1
acciones_cl,2016-10-03,2026-10-07,1757,41.1
oro,2016-10-03,2026-10-08,1756,41.1
cobre,2016-10-03,2026-10-08,1756,41.1
sp500,2016-10-03,2026-10-08,1756,41.1
dolar,2010-01-04,2026-07-31,145,3.4


Recortemos al período en que **todas** las series existen, y miremos una semana con feriados en Chile:

In [9]:
inicio = diagnostico['primer dato'].max()     # la serie que empieza más tarde
fin    = diagnostico['último dato'].min()     # la serie que termina más temprano
comun = diarios.loc[inicio:fin]              # copia recortada; 'diarios' conserva toda la historia
print(f'Período común: {inicio:%Y-%m-%d} a {fin:%Y-%m-%d}')

comun.loc['2025-09-15':'2025-09-23']

Período común: 2016-10-03 a 2026-07-31


,brent,acciones_cl,oro,cobre,sp500,dolar
2025-09-15,67.440002,33.156384,3719.000000,4.6555,6615.279785,954.58
2025-09-16,68.470001,33.117142,3725.100098,4.6330,6606.759766,951.14
2025-09-17,67.949997,32.764004,3717.800049,4.5710,6600.350098,948.57
2025-09-18,67.440002,32.960194,3678.300049,4.5410,6631.959961,NaN
2025-09-19,66.680000,32.950386,3705.800049,4.5690,6664.359863,NaN
2025-09-22,66.570000,32.862099,3775.100098,4.5730,6693.750000,951.03
2025-09-23,67.629997,33.038673,3815.699951,4.5845,6656.919922,954.72


### ¿Qué hacer con los faltantes?

| Estrategia | Código | Cuándo sirve | Riesgo |
|:---|:---|:---|:---|
| **Eliminar filas** | `dropna()` | Comparar series día a día (retornos, correlaciones) usando solo días en que todas tienen dato | Se pierden observaciones |
| **Arrastrar el último valor** | `ffill()` | Precios: el último precio conocido sigue vigente mientras el mercado está cerrado | Si el hueco es largo, se arrastra un dato viejo (usar `limit=`) |
| **Interpolar** | `interpolate()` | Dibujar o describir el pasado | **Usa el dato siguiente**: nunca para pronosticar (look-ahead bias, cuaderno U2-05) |
| **Agregar a menor frecuencia** | `resample('MS').mean()` | Pasar de diario a mensual: el promedio ignora los `NaN` | Se pierde detalle diario |

Comparemos las tres primeras sobre el dólar en Fiestas Patrias 2025:

In [10]:
semana = slice('2025-09-15', '2025-09-23')
pd.DataFrame({
    'original':      comun['dolar'],
    'ffill':         comun['dolar'].ffill(),
    'interpolate':   comun['dolar'].interpolate(),
}).loc[semana]

,original,ffill,interpolate
2025-09-15,954.58,954.58,954.58
2025-09-16,951.14,951.14,951.14
2025-09-17,948.57,948.57,948.57
2025-09-18,NaN,948.57,949.39
2025-09-19,NaN,948.57,950.21
2025-09-22,951.03,951.03,951.03
2025-09-23,954.72,954.72,954.72


In [11]:
print(f'Filas del período común:  {len(comun)}')
print(f'Filas tras dropna():      {len(comun.dropna())}   (solo días con todas las series)')
print(f'Faltantes tras ffill():   {comun.ffill().isna().sum().sum()}')

Filas del período común:  2539
Filas tras dropna():      2371   (solo días con todas las series)
Faltantes tras ffill():   0


> **Decisión para este cuaderno:** para comparar series **diarias** usaremos solo los días con todas las series (`dropna`);
> para pasar a **mensual** usaremos `resample`, que ignora los faltantes. Lo importante es **decidir a propósito y dejarlo escrito**.

### De diario a mensual: ¿promedio o cierre?

`resample('MS')` agrupa por mes (`'MS'` = *month start*: cada mes queda fechado el día 1). Hay que elegir cómo resumir:

* **Promedio del mes** (`.mean()`): representa el nivel típico del mes. Es lo que se usa para indicadores macro (ej. "dólar promedio de julio").
* **Último dato del mes** (`.last()`): el precio al cierre. Es lo que se usa para calcular **retornos** de inversiones.

In [12]:
# Solo hasta el último día común ('fin'), para no mezclar meses incompletos
prom_mensual   = diarios.loc[:fin].resample('MS').mean()
cierre_mensual = diarios.loc[:fin].resample('MS').last()

pd.DataFrame({'dólar promedio del mes': prom_mensual['dolar'],
              'dólar al cierre del mes': cierre_mensual['dolar']}).tail(6).round(1)

,dólar promedio del mes,dólar al cierre del mes
2026-02-01,862.0,861.2
2026-03-01,909.9,931.6
2026-04-01,897.9,901.8
2026-05-01,897.6,892.9
2026-06-01,903.4,922.3
2026-07-01,931.0,924.8


In [13]:
# Tabla mensual de trabajo: promedios de mercado + índices de precios y actividad
mensual = prom_mensual.join(mensuales, how='outer')
mensual.tail(4).round(2)

,brent,acciones_cl,oro,cobre,sp500,dolar,ipc_canasta_2018,ipc_canasta_2023,ipc_cl,ipc_eeuu,imacec
2026-05-01,104.09,40.26,4598.23,6.26,7412.55,897.64,NaN,112.37,112.37,333.98,112.66
2026-06-01,84.62,39.80,4257.85,6.32,7450.03,903.38,NaN,112.35,112.35,332.57,111.03
2026-07-01,84.51,39.36,4078.74,6.28,7481.34,930.97,NaN,112.45,112.45,NaN,NaN
2026-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,113.15,113.15,NaN,NaN


## Anexo: los mismos datos, pedidos "a mano" con `requests`

`yfinance` consulta una dirección web de Yahoo. Podemos pedirla directamente con `requests`, igual que hicimos con la
API del Banco Mundial en U2-01. Sirve para entender qué hace la librería por dentro y como plan B si deja de funcionar.

La diferencia: las fechas vienen en **segundos desde 1970** y hay que convertirlas, y el precio no viene ajustado por dividendos.

In [26]:
url = 'https://query1.finance.yahoo.com/v8/finance/chart/HG=F'
respuesta = requests.get(url, params={'range': '1mo', 'interval': '1d'},
                         headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
datos = respuesta.json()['chart']['result'][0]

fechas = pd.to_datetime(datos['timestamp'], unit='s')            # segundos -> fechas
cierre = datos['indicators']['quote'][0]['close']                # precios de cierre

pd.Series(cierre, index=fechas, name='cobre').tail()

2026-10-02 04:00:00    6.4920
2026-10-05 04:00:00    6.5855
2026-10-06 04:00:00    6.5945
2026-10-07 04:00:00    6.5965
2026-10-08 04:00:00    6.6090
Name: cobre, dtype: float64